# Loss function

_Machine Learning_

**One number that says how wrong a single prediction was.**

A loss function measures how wrong one prediction is.

     It compares the prediction to the true answer.

---

This notebook is a practice scaffold for the **Loss function** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. Each row is one example; the columns are its features, and the target is a continuous value.

In [ ]:
from sklearn.datasets import load_diabetes

data = load_diabetes(as_frame=True)
print("rows x columns:", data.frame.shape)
print("feature columns:", list(data.data.columns))
print("target summary:")
print(data.target.describe())
data.frame.head()

## Reference implementation — NumPy

### Step 1 — Squared loss for a regression prediction

Squared loss measures how far a continuous prediction `z` lands from the true value `y`. We use the convention `1/2 (y - z)^2`; the `1/2` is a tidy factor that makes the derivative come out as just `(z - y)`. The larger the gap, the larger the loss — and because of the square, it grows *quadratically*.

In [ ]:
# squared loss: 1/2 (y - z)^2
y = 3.0          # true value
z = 2.5          # model's prediction
squared_loss = 0.5 * (y - z) ** 2
print("squared loss:", squared_loss)

### Step 2 — Hinge loss for a margin classifier

The hinge loss is used by SVM-style classifiers where the label is `+1` or `-1`. It looks at the **margin** `y * z` — how confidently and correctly the score points to the right class. If the margin reaches `1` the loss is zero; otherwise the loss is `1 - margin`, so confident-but-wrong predictions are penalised.

In [ ]:
# hinge loss (SVM): max(0, 1 - margin), margin = y * z, y in {-1, +1}
y_pm = 1.0       # true label, encoded as +1 / -1
score = 0.3      # raw model score z
margin = y_pm * score
hinge_loss = max(0.0, 1 - margin)
print("hinge loss:", hinge_loss)

### Step 3 — Binary cross-entropy for a probability

When the model outputs a probability `p` for the positive class, cross-entropy is the natural loss. For a positive example (`y = 1`) it reduces to `-log(p)`: the closer `p` is to 1 the smaller the loss, and a confident wrong answer is punished sharply because `log` heads toward minus infinity near 0.

In [ ]:
# binary cross-entropy: -[y log p + (1-y) log(1-p)]
y_bin = 1.0      # true label, 0 or 1
p = 0.88         # model's predicted probability for the positive class
bce = -(y_bin * np.log(p) + (1 - y_bin) * np.log(1 - p))
cross_entropy = round(float(bce), 4)
print("cross-entropy:", cross_entropy)

### Step 4 — Watch squared loss grow with the gap

Because squared loss squares the error, doubling the gap *quadruples* the loss. The loop below makes that concrete: a gap of 1, 2, then 4 produces losses of 0.5, 2.0, and 8.0. This is why squared loss reacts so strongly to large outliers.

In [ ]:
# bigger gap -> much bigger squared loss (4x the loss for 2x the gap)
for gap in [1.0, 2.0, 4.0]:
    loss_for_gap = 0.5 * gap ** 2
    print("gap %.0f -> squared loss %.1f" % (gap, loss_for_gap))

## Visualize the data & results

_On real diabetes-progression predictions, how hard does each loss punish a prediction error as it grows?_

### Step 1 — Fit a real model and collect its errors

We train a plain linear regression on 442 real diabetes patients, where the target is disease progression. After fitting on the training split we predict on the held-out test split and compute the **residuals** — the actual prediction errors `y - prediction` that we'll feed into the loss functions.

In [ ]:
# 442 real diabetes patients, target = disease progression
from sklearn.datasets import load_diabetes
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

db = load_diabetes()
Xtr, Xte, ytr, yte = train_test_split(db.data, db.target, test_size=0.3, random_state=0)
model = LinearRegression().fit(Xtr, ytr)
predictions = model.predict(Xte)
residuals = yte - predictions        # real prediction errors

### Step 2 — Compare how squared and absolute loss grow

To see the *shape* of each loss, we sweep a range of error values `r` and evaluate squared loss `0.5 r^2` and absolute loss `|r|` at each point. Squared loss curves upward and quickly dominates, while absolute loss rises in a straight line — squared loss cares far more about big errors.

In [ ]:
# how each loss grows across the real residual range
r = np.linspace(-100, 100, 25)
squared = 0.5 * r ** 2
absolute = np.abs(r)

### Step 3 — Plot the curves and three real residuals

The left panel draws both loss curves so you can see squared loss overtaking absolute loss. The right panel picks a small, a median, and a large real residual and shows the squared loss for each — concrete proof that a big miss costs disproportionately more.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

# Left — loss shape vs prediction error
ax1.plot(r, squared, color="#ffb454", label="squared 0.5 r squared")
ax1.plot(r, absolute, color="#4ea1ff", label="absolute |r|")
ax1.set_xlabel("prediction error r")
ax1.set_ylabel("loss")
ax1.set_title("Loss vs prediction error")
ax1.legend()

# Right — squared loss at three real residual sizes
sorted_abs_residuals = np.sort(np.abs(residuals))
small = sorted_abs_residuals[5]
median = sorted_abs_residuals[len(residuals) // 2]
large = sorted_abs_residuals[-3]
ex = np.array([small, median, large])
ax2.bar(["small", "median", "large"], 0.5 * ex ** 2,
        color=["#4ea1ff", "#ffb454", "#ff7b72"])
ax2.set_title("Squared loss at three real residual sizes")

plt.show()